# Seminar 10: Vector quantizing

Thanks to [lucidrains](https://github.com/lucidrains) for generously providing [implementation for vector quantizing and residual vector quantizing](https://github.com/lucidrains/vector-quantize-pytorch/tree/master)!

Choose Encodec if you need:
* High-fidelity audio compression for both speech and music.
* Integration with audio generation tools like AudioCraft.
* Support for stereo audio at higher sampling rates. 24 kHz (mono), 32 kHz (mono), 48 kHz (stereo)

Choose SoundStream if you need:
* Low-latency, real-time audio processing on devices with limited computational resources.
* A single model capable of operating across multiple bitrates.
* Joint compression and enhancement capabilities, such as noise suppression. - only 24 kHz (mono)

# Encodec

![img](./images/encodec.png)

In [2]:
from datasets import load_dataset, Audio
from transformers import EncodecModel, AutoProcessor

In [3]:
# dummy dataset, however you can swap this with an dataset on the 🤗 hub or bring your own
librispeech_dummy = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")

# load the model + processor (for pre-processing the audio)
model = EncodecModel.from_pretrained("facebook/encodec_24khz")
processor = AutoProcessor.from_pretrained("facebook/encodec_24khz")

# cast the audio data to the correct sampling rate for the model
librispeech_dummy = librispeech_dummy.cast_column("audio", Audio(sampling_rate=processor.sampling_rate))
audio_sample = librispeech_dummy[0]["audio"]["array"]

# pre-process the inputs
inputs = processor(raw_audio=audio_sample, sampling_rate=processor.sampling_rate, return_tensors="pt")

# explicitly encode then decode the audio inputs
encoder_outputs = model.encode(inputs["input_values"], inputs["padding_mask"])
audio_values = model.decode(encoder_outputs.audio_codes, encoder_outputs.audio_scales, inputs["padding_mask"])[0]

# or the equivalent with a forward pass
audio_values = model(inputs["input_values"], inputs["padding_mask"]).audio_values

# you can also extract the discrete codebook representation for LM tasks
# output: concatenated tensor of all the representations
audio_codes = model(inputs["input_values"], inputs["padding_mask"]).audio_codes

README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

The image processor of type `EncodecImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
`use_fast` is set to `True` but the image processor class does not have a fast version.  Falling back to the slow version.


In [11]:
display(Audio(audio_values[0].cpu().detach().numpy()))

Audio(sampling_rate=array([[ 3.0870596e-04, -2.2500251e-04, -2.7547700e-05, ...,
        -3.5534546e-04,  5.8134360e-04,  1.0407589e-03]],
      shape=(1, 140520), dtype=float32), decode=True, num_channels=None, stream_index=None)

# Soundstream

In [16]:
from __future__ import annotations

import random
from math import ceil
from functools import partial, cache
from itertools import zip_longest

import torch
from torch import nn, Tensor
from torch.nn import Module, ModuleList
import torch.nn.functional as F
import torch.distributed as dist
# from vector_quantize_pytorch.vector_quantize_pytorch import VectorQuantize

from einops import rearrange, repeat, reduce, pack, unpack

from einx import get_at

![img](./images/soundstream.png)

## RVQ
[Good blockpost about RVQ](https://www.assemblyai.com/blog/what-is-residual-vector-quantization#:~:text=RVQ%20breaks%20down%20the%20quantization,scaling%20the%20number%20of%20layers)

Vector quantization:
![img](./images/vq.jpeg)

Residual vector quantization:
![img](./images/rvq.jpeg)

![img](./images/llm_quant.png)

In [49]:
from __future__ import annotations

import random
from math import ceil
from functools import partial, cache
from itertools import zip_longest

import torch
from torch import nn, Tensor
from torch.nn import Module, ModuleList
import torch.nn.functional as F
import torch.distributed as dist
from vector_quantize_pytorch.vector_quantize_pytorch import VectorQuantize

from einops import rearrange, repeat, reduce, pack, unpack

from einx import get_at

# helper functions

def exists(val):
    return val is not None

def first(it):
    return it[0]

def default(val, d):
    return val if exists(val) else d

def cast_tuple(t, length = 1):
    return t if isinstance(t, tuple) else ((t,) * length)

def unique(arr):
    return list({*arr})

def round_up_multiple(num, mult):
    return ceil(num / mult) * mult

# distributed helpers

def is_distributed():
    return dist.is_initialized() and dist.get_world_size() > 1

def get_maybe_sync_seed(device, max_size = 10_000):
    rand_int = torch.randint(0, max_size, (), device = device)

    if is_distributed():
        dist.all_reduce(rand_int)

    return rand_int.item()

In [41]:
class ResidualVQ(nn.Module):
    """
    Residual Vector Quantization (RVQ).

    Core idea:
        We do not quantize x in one shot with one codebook.
        Instead, we quantize it in stages.

        Stage 1:
            q1 ≈ x
            residual_1 = x - q1

        Stage 2:
            q2 ≈ residual_1
            residual_2 = residual_1 - q2

        Stage 3:
            q3 ≈ residual_2
            ...

        Final reconstruction:
            x_hat = q1 + q2 + q3 + ...

    Why this is useful:
        - Each quantizer only needs to model what previous quantizers missed.
        - This usually gives much better rate-distortion tradeoff than one big VQ.
        - It is the standard idea behind RVQ / stacked quantization.

    Important assumptions:
        - This class wraps several `VectorQuantize` modules.
        - Helper functions like `exists`, `default`, `cast_tuple`, `first`,
          `unique`, `pack`, `unpack`, `rearrange`, `reduce`, `get_at`,
          `round_up_multiple`, and `get_maybe_sync_seed`
          are assumed to exist elsewhere in the codebase.
        - `VectorQuantize` is assumed to support:
              vq(x, indices=None, mask=None, ...)
          and return either:
              (quantized, embed_indices, loss)                # normal mode
          or:
              (quantized, ce_loss)                            # teacher-forced / loss mode

    Typical tensor shapes in sequence mode:
        x:              [B, N, D]
            B = batch size
            N = number of tokens / time steps
            D = feature dimension

        all_indices:    [B, N, Q]
            Q = number of residual quantizers

        all_codes:      [Q, B, N, D_codebook]

    If `accept_image_fmap=True`, the wrapped VQ can accept image feature maps too.
    Then indices are typically shaped like:
        [B, H, W, Q]
    """

    def __init__(
        self,
        *,
        dim,
        num_quantizers: int | None = None,
        codebook_size: int | tuple[int, ...],
        codebook_dim=None,
        shared_codebook=False,
        heads=1,
        quantize_dropout=False,
        quantize_dropout_cutoff_index=0,
        quantize_dropout_multiple_of=1,
        accept_image_fmap=False,
        implicit_neural_codebook=False,  # QINCo-style codebook conditioning
        mlp_kwargs: dict = dict(),
        **vq_kwargs
    ):
        super().__init__()

        # ---------------------------------------------------------------------
        # Basic sanity checks
        # ---------------------------------------------------------------------
        # This implementation only supports one code stream per token.
        # Multi-headed VQ would require different residual bookkeeping.
        assert heads == 1, 'residual vq is not compatible with multi-headed codes'

        # Either:
        #   - the number of quantizers is explicitly given
        # or:
        #   - codebook_size is a tuple, one size per quantizer
        assert exists(num_quantizers) or isinstance(codebook_size, tuple)

        # ---------------------------------------------------------------------
        # Input/output projection around the codebook space
        # ---------------------------------------------------------------------
        # `dim`               = model feature dimension coming into ResidualVQ
        # `codebook_dim`      = dimension used inside each VQ codebook
        #
        # If dim != codebook_dim, we project in before quantization and
        # project out after quantization.
        codebook_dim = default(codebook_dim, dim)
        codebook_input_dim = codebook_dim * heads  # heads is always 1 here

        requires_projection = codebook_input_dim != dim

        # Project model features into codebook space if needed
        self.project_in = nn.Linear(dim, codebook_input_dim) if requires_projection else nn.Identity()

        # Project summed quantized vectors back to original model space if needed
        self.project_out = nn.Linear(codebook_input_dim, dim) if requires_projection else nn.Identity()

        self.has_projections = requires_projection
        self.accept_image_fmap = accept_image_fmap
        self.implicit_neural_codebook = implicit_neural_codebook

        # ---------------------------------------------------------------------
        # Optional QINCo-style implicit neural codebook setup
        # ---------------------------------------------------------------------
        # If enabled, the codebook is not treated as a plain fixed lookup table.
        # Instead, later-stage code vectors can be transformed by an MLP
        # conditioned on the current partial reconstruction.
        #
        # That means:
        #   code(stage k) = MLP_k(raw_codebook_entries, condition=quantized_so_far)
        #
        # This makes the effective codebook context-dependent.
        if implicit_neural_codebook:
            vq_kwargs.update(
                learnable_codebook=True,
                ema_update=False
            )

        # ---------------------------------------------------------------------
        # Shared-codebook mode
        # ---------------------------------------------------------------------
        # In shared-codebook mode, all RVQ layers point to the SAME codebook
        # object. They are still used at different residual stages, but the
        # entries themselves are shared.
        #
        # Because multiple layers touch the same codebook, the wrapped VQ
        # needs manual EMA / optimizer updates, which are performed once at end.
        if shared_codebook:
            vq_kwargs.update(
                manual_ema_update=True,
                manual_in_place_optimizer_update=True
            )

        # ---------------------------------------------------------------------
        # Codebook size per stage
        # ---------------------------------------------------------------------
        # `codebook_size` can be:
        #   - one int: same size for all quantizers
        #   - tuple[int, ...]: possibly different size per quantizer
        codebook_sizes = cast_tuple(codebook_size, num_quantizers)

        num_quantizers = default(num_quantizers, len(codebook_sizes))
        assert len(codebook_sizes) == num_quantizers

        self.num_quantizers = num_quantizers
        self.codebook_sizes = codebook_sizes

        # True if every stage uses the same number of code entries
        self.uniform_codebook_size = len(unique(codebook_sizes)) == 1

        # ---------------------------------------------------------------------
        # Build the stack of quantizers
        # ---------------------------------------------------------------------
        # Each layer quantizes the current residual.
        #
        # Important:
        # Each `VectorQuantize` here is created WITHOUT its own internal
        # projections. ResidualVQ handles the in/out projections globally.
        self.layers = ModuleList([
            VectorQuantize(
                dim=codebook_dim,
                codebook_size=layer_codebook_size,
                codebook_dim=codebook_dim,
                accept_image_fmap=accept_image_fmap,
                **vq_kwargs
            )
            for layer_codebook_size in codebook_sizes
        ])

        assert all([not vq.has_projections for vq in self.layers])

        # ---------------------------------------------------------------------
        # Quantizer dropout
        # ---------------------------------------------------------------------
        # This is the RVQ version of structured dropout used in codecs like
        # Encodec / SoundStream style systems.
        #
        # During training, we randomly stop after some quantizer index k and
        # drop all finer quantizers k+1, k+2, ...
        #
        # Why:
        #   The decoder learns to reconstruct from coarse-only codes too.
        #   This improves robustness and allows variable bitrate behavior.
        self.quantize_dropout = quantize_dropout and num_quantizers > 1

        assert quantize_dropout_cutoff_index >= 0
        self.quantize_dropout_cutoff_index = quantize_dropout_cutoff_index

        # If this is > 1, the dropout boundary is rounded up to a multiple.
        # Example:
        #   multiple_of = 4
        #   then you only keep 4, 8, 12, ... quantizers, never 5 or 6.
        self.quantize_dropout_multiple_of = quantize_dropout_multiple_of

        # ---------------------------------------------------------------------
        # MLPs for implicit neural codebooks
        # ---------------------------------------------------------------------
        # There is no MLP for the first quantizer.
        # Later quantizers may transform their codebook entries conditioned on
        # the running partial reconstruction.
        self.mlps = None

        if implicit_neural_codebook:
            self.mlps = ModuleList([
                MLP(
                    dim=codebook_dim,
                    l2norm_output=first(self.layers).use_cosine_sim,
                    **mlp_kwargs
                )
                for _ in range(num_quantizers - 1)
            ])
        else:
            self.mlps = (None,) * (num_quantizers - 1)

        # ---------------------------------------------------------------------
        # Shared codebook wiring
        # ---------------------------------------------------------------------
        # If shared_codebook=True, all layers literally reference the same
        # underlying codebook tensor / module.
        self.shared_codebook = shared_codebook

        if not shared_codebook:
            return

        # Shared codebook only makes sense cleanly when all stages have the
        # same codebook size.
        assert self.uniform_codebook_size

        first_vq, *rest_vq = self.layers
        codebook = first_vq._codebook

        for vq in rest_vq:
            vq._codebook = codebook

    # -------------------------------------------------------------------------
    # Convenience properties
    # -------------------------------------------------------------------------
    @property
    def codebook_size(self):
        """
        Convenience accessor.

        Note:
            If codebook sizes differ across stages, this returns ONLY the
            first layer's size. In that case, `self.codebook_sizes` is the
            safer thing to inspect.
        """
        return self.layers[0].codebook_size

    @property
    def codebook_dim(self):
        """Dimension of each code vector used by one quantizer stage."""
        return self.layers[0].codebook_dim

    @property
    def codebooks(self):
        """
        Returns the raw codebook embeddings.

        If all stages use the same size:
            returns tensor of shape [Q, C, D]
                Q = num quantizers
                C = codebook size
                D = codebook dim

        If sizes differ across stages:
            returns tuple of tensors, one per stage,
            because stacking is no longer possible.

        Internal note:
            Each layer stores codebook embeddings with an extra leading
            singleton dimension; `rearrange(codebook, '1 ... -> ...')`
            removes it.
        """
        codebooks = [layer._codebook.embed for layer in self.layers]
        codebooks = tuple(rearrange(codebook, '1 ... -> ...') for codebook in codebooks)

        if not self.uniform_codebook_size:
            return codebooks

        codebooks = torch.stack(codebooks)
        return codebooks

    # -------------------------------------------------------------------------
    # Reconstruction utilities
    # -------------------------------------------------------------------------
    def get_codes_from_indices(self, indices):
        """
        Convert discrete RVQ indices back into the actual code vectors.

        Input:
            indices:
                sequence mode:    [B, N, Q_used] or [B, N, Q]
                image mode:       [B, H, W, Q_used] or [B, H, W, Q]

            Q_used may be < self.num_quantizers if:
                - the signal was encoded with quantizer dropout
                - or we intentionally only kept coarse quantizers

        Output:
            all_codes:
                sequence mode:    [Q, B, N, D_codebook]
                image mode:       [Q, B, H, W, D_codebook]

            Each slice all_codes[q] contains the vectors chosen from
            quantizer q's codebook.
        """
        batch, quantize_dim = indices.shape[0], indices.shape[-1]

        # Flatten all non-batch, non-quantizer axes into one axis so we can
        # handle both [B, N, Q] and [B, H, W, Q] with one code path.
        #
        # After packing:
        #   indices -> [B, T, Q]
        # where T is "all positions per sample" (tokens or pixels).
        indices, ps = pack([indices], 'b * q')

        # -------------------------------------------------------------
        # Support reconstruction from a coarser signal
        # -------------------------------------------------------------
        # If fewer quantizers were stored than this model actually has,
        # pad the missing stages with -1.
        #
        # Convention:
        #   index == -1 means "this quantizer was dropped / not used"
        #
        # That happens with quantizer dropout or variable-rate coding.
        if quantize_dim < self.num_quantizers:
            assert self.quantize_dropout > 0., (
                'quantize dropout must be greater than 0 if you wish to '
                'reconstruct from a signal with less fine quantizations'
            )
            indices = F.pad(indices, (0, self.num_quantizers - quantize_dim), value=-1)

        # -------------------------------------------------------------
        # Handle dropped quantizers
        # -------------------------------------------------------------
        # We cannot use -1 directly for indexing, so:
        #   1) record a mask of dropped positions
        #   2) temporarily replace -1 with 0
        #   3) gather dummy code 0
        #   4) zero those gathered vectors out afterward
        mask = indices == -1
        indices = indices.masked_fill(mask, 0)

        # -------------------------------------------------------------
        # Fast path: all codebooks same size and no implicit codebook
        # -------------------------------------------------------------
        # If codebooks are uniform and static, we can gather everything in one go.
        if not self.implicit_neural_codebook and self.uniform_codebook_size:
            # self.codebooks: [Q, C, D]
            # indices:        [B, T, Q]
            # result:         [Q, B, T, D]
            all_codes = get_at('q [c] d, b n q -> q b n d', self.codebooks, indices)

        else:
            # ---------------------------------------------------------
            # Slow/general path
            # ---------------------------------------------------------
            # Needed if:
            #   - codebook sizes differ across stages
            #   - or codebook entries are transformed dynamically by MLPs
            #
            # `quantized_out` is the running partial reconstruction used as the
            # condition for implicit neural codebooks.
            code_transform_mlps = (None, *self.mlps)
            all_codes = []
            quantized_out = 0.

            for codes, indices_per_stage, maybe_transform_mlp in zip(
                self.codebooks,
                indices.unbind(dim=-1),   # iterate over Q stages
                code_transform_mlps
            ):
                # If using implicit neural codebooks, transform the codebook
                # conditioned on what previous stages have already reconstructed.
                if exists(maybe_transform_mlp):
                    # Expected output shape is compatible with get_at below.
                    codes = maybe_transform_mlp(codes, condition=quantized_out)

                    # codes:            [B, T, C, D]   or compatible broadcast form
                    # indices_per_stage:[B, T]
                    # layer_codes:      [B, T, D]
                    layer_codes = get_at('b n [c] d, b n -> b n d', codes, indices_per_stage)
                else:
                    # Static codebook case:
                    # codes:            [C, D]
                    # indices_per_stage:[B, T]
                    # layer_codes:      [B, T, D]
                    layer_codes = get_at('[c] d, b n -> b n d', codes, indices_per_stage)

                all_codes.append(layer_codes)

                # Running partial reconstruction used as condition for later stages
                quantized_out += layer_codes

            # [Q, B, T, D]
            all_codes = torch.stack(all_codes)

        # -------------------------------------------------------------
        # Zero out codes that corresponded to dropped quantizers
        # -------------------------------------------------------------
        # mask:      [B, T, Q]
        # rearrange: [Q, B, T, 1]
        all_codes = all_codes.masked_fill(rearrange(mask, 'b n q -> q b n 1'), 0.)

        # Restore original token/spatial layout.
        # Example:
        #   [Q, B, T, D] -> [Q, B, N, D]
        #   [Q, B, T, D] -> [Q, B, H, W, D]
        all_codes, = unpack(all_codes, ps, 'q b * d')

        return all_codes

    def get_output_from_indices(self, indices):
        """
        Reconstruct the final quantized output directly from RVQ indices.

        Steps:
            1) map indices -> code vectors per stage
            2) sum across residual quantizer stages
            3) project back to model dimension if needed
        """
        codes = self.get_codes_from_indices(indices)         # [Q, ... , D_codebook]
        codes_summed = reduce(codes, 'q ... -> ...', 'sum') # [..., D_codebook]
        return self.project_out(codes_summed)               # [..., D_model]

    # -------------------------------------------------------------------------
    # Main forward
    # -------------------------------------------------------------------------
    def forward(
        self,
        x,
        mask=None,
        indices: Tensor | list[Tensor] | None = None,
        return_all_codes=False,
        sample_codebook_temp=None,
        freeze_codebook=False,
        rand_quantize_dropout_fixed_seed=None
    ):
        """
        Forward modes
        =============

        1) Normal encoding mode (indices is None)
           -------------------------------------
           The module:
               - quantizes x stage by stage
               - returns quantized output, indices, and VQ losses

           Returns:
               quantized_out, all_indices, all_losses
           or if return_all_codes=True:
               quantized_out, all_indices, all_losses, all_codes

        2) Teacher-forced / loss mode (indices is given)
           ---------------------------------------------
           The module uses provided target indices and returns cross-entropy
           style loss from the underlying quantizers.

           Returns:
               quantized_out, total_ce_loss

        Inputs
        ------
        x:
            Usually [B, N, D] in sequence mode.

        mask:
            Optional mask forwarded to each VectorQuantize layer.

        indices:
            Optional ground-truth code indices for each stage.
            Used when training an autoregressive / entropy model over codes,
            or when asking the quantizer to compute CE loss against known codes.

        return_all_codes:
            If True, also return the actual code vectors chosen at every stage.

        sample_codebook_temp:
            Optional temperature for stochastic code sampling in the wrapped VQ.

        freeze_codebook:
            If True, do not update the codebook during this forward pass.

        rand_quantize_dropout_fixed_seed:
            Optional seed so quantizer-dropout boundary is reproducible across
            devices / processes.
        """
        num_quant = self.num_quantizers
        quant_dropout_multiple_of = self.quantize_dropout_multiple_of
        return_loss = exists(indices)   # True => use teacher-forced / CE mode
        device = x.device

        # Map model features into codebook feature space if needed
        x = self.project_in(x)

        # Current implementation does not support passing teacher-forced indices
        # when in image feature map mode.
        assert not (self.accept_image_fmap and exists(indices))

        # `quantized_out` is the running partial reconstruction q1 + q2 + ...
        # `residual` is what remains to be explained by later quantizers
        quantized_out = 0.
        residual = x

        all_losses = []
        all_indices = []

        # Allow list[Tensor] input for convenience; convert to one tensor.
        if isinstance(indices, list):
            indices = torch.stack(indices)

        if return_loss:
            # In CE mode, dropped indices (-1) are not allowed.
            # The CE target must fully specify the stage index at every position.
            assert not torch.any(indices == -1), (
                'some of the residual vq indices were dropped out. '
                'please use indices derived when the module is in eval mode '
                'to derive cross entropy loss'
            )
            ce_losses = []

        # Quantizer dropout is only used during normal training-time quantization,
        # not when computing CE with provided indices.
        should_quantize_dropout = self.training and self.quantize_dropout and not return_loss

        # ---------------------------------------------------------------------
        # Sample the "last active quantizer" for quantizer dropout
        # ---------------------------------------------------------------------
        if should_quantize_dropout:
            # If caller did not provide a seed, generate one that can be synced
            # across devices if needed.
            if not exists(rand_quantize_dropout_fixed_seed):
                rand_quantize_dropout_fixed_seed = get_maybe_sync_seed(device)

            rand = random.Random(rand_quantize_dropout_fixed_seed)

            # Pick a stage after which all remaining finer quantizers are dropped.
            rand_quantize_dropout_index = rand.randrange(
                self.quantize_dropout_cutoff_index, num_quant
            )

            # Optionally force the kept-quantizer count to be a multiple.
            # Example:
            #   if chosen last active stage is 5 and multiple_of=4,
            #   we round up so last active stage becomes 7,
            #   meaning we keep quantizers 0..7.
            if quant_dropout_multiple_of != 1:
                rand_quantize_dropout_index = (
                    round_up_multiple(rand_quantize_dropout_index + 1, quant_dropout_multiple_of) - 1
                )

            # Shape of placeholder indices for dropped stages:
            #   sequence mode: [B, N]
            #   image mode:    [B, H, W]
            null_indices_shape = (
                (x.shape[0], *x.shape[-2:])
                if self.accept_image_fmap
                else tuple(x.shape[:2])
            )

            # -1 is the sentinel meaning "dropped stage / no code"
            null_indices = torch.full(
                null_indices_shape,
                -1.,
                device=device,
                dtype=torch.long
            )

            # Dropped stages contribute zero VQ loss
            null_loss = torch.full((1,), 0., device=device, dtype=x.dtype)

        # ---------------------------------------------------------------------
        # Setup codebook transforms for implicit neural codebooks
        # ---------------------------------------------------------------------
        maybe_code_transforms = (None,) * len(self.layers)
        if self.implicit_neural_codebook:
            maybe_code_transforms = (None, *self.mlps)

        # Save inputs to each stage.
        # Needed later for shared-codebook expiration / EMA maintenance.
        all_residuals = []

        # ---------------------------------------------------------------------
        # Main RVQ loop
        # ---------------------------------------------------------------------
        for quantizer_index, (vq, maybe_mlp) in enumerate(zip(self.layers, maybe_code_transforms)):

            # -------------------------------------------------------------
            # Quantizer dropout branch:
            # If this stage is beyond the chosen last active stage, skip it.
            # -------------------------------------------------------------
            if should_quantize_dropout and quantizer_index > rand_quantize_dropout_index:
                all_indices.append(null_indices)
                all_losses.append(null_loss)
                continue

            # In teacher-forced CE mode, use provided target indices for this stage
            layer_indices = None
            if return_loss:
                layer_indices = indices[..., quantizer_index]

            # If using implicit neural codebooks, bind current partial
            # reconstruction as conditioning input.
            if exists(maybe_mlp):
                maybe_mlp = partial(maybe_mlp, condition=quantized_out)

            # Save the current residual BEFORE quantizing it
            all_residuals.append(residual)

            # -------------------------------------------------------------
            # Quantize current residual
            # -------------------------------------------------------------
            # Normal mode:
            #   returns quantized, embed_indices, loss
            #
            # CE mode:
            #   returns quantized, ce_loss
            quantized, *rest = vq(
                residual,
                mask=mask,
                indices=layer_indices,
                sample_codebook_temp=sample_codebook_temp,
                freeze_codebook=freeze_codebook,
                codebook_transform_fn=maybe_mlp
            )

            # -------------------------------------------------------------
            # Update residual and running reconstruction
            # -------------------------------------------------------------
            # This line is one of the most important in RVQ:
            #
            #   residual_next = residual_current - quantized
            #
            # But notice the `.detach()`.
            #
            # Why detach here?
            #   Later quantizers should see the numerical residual left by
            #   earlier stages, but we do NOT want gradients from later stages
            #   to flow backward through this subtraction path into earlier
            #   quantized outputs.
            #
            # In other words:
            #   - the residual value depends on earlier quantizers
            #   - but its gradient path is cut
            residual = residual - quantized.detach()

            # Running sum of all stage reconstructions
            quantized_out = quantized_out + quantized

            # -------------------------------------------------------------
            # CE mode: collect CE losses and continue
            # -------------------------------------------------------------
            if return_loss:
                ce_loss = rest[0]
                ce_losses.append(ce_loss)
                continue

            # -------------------------------------------------------------
            # Normal mode: collect indices and VQ losses
            # -------------------------------------------------------------
            embed_indices, loss = rest
            all_indices.append(embed_indices)
            all_losses.append(loss)

        # ---------------------------------------------------------------------
        # Shared-codebook maintenance
        # ---------------------------------------------------------------------
        # If all stages share the same codebook, update that shared codebook ONCE
        # after the full RVQ pass.
        #
        # Why not per stage?
        #   Because every stage points to the same codebook object.
        if self.training and self.shared_codebook:
            shared_layer = first(self.layers)

            # Update EMA statistics of the shared codebook
            shared_layer._codebook.update_ema()

            # Apply optimizer step if codebook uses in-place optimizer updates
            shared_layer.update_in_place_optimizer()

            # Expire / refresh underused codes using all stage inputs
            shared_layer.expire_codes_(torch.cat(all_residuals, dim=-2))

        # Project summed quantized vectors back to model dimension if needed
        quantized_out = self.project_out(quantized_out)

        # ---------------------------------------------------------------------
        # Return CE mode output
        # ---------------------------------------------------------------------
        if return_loss:
            # Teacher-forced mode returns only:
            #   - quantized output reconstructed from provided indices
            #   - sum of CE losses over stages
            return quantized_out, sum(ce_losses)

        # ---------------------------------------------------------------------
        # Normal mode output
        # ---------------------------------------------------------------------
        # Stack along a new last axis = quantizer index Q
        #
        # Sequence example:
        #   all_indices: [B, N, Q]
        #
        # Losses:
        #   exact shape depends on the wrapped VQ implementation,
        #   but the last dimension is always quantizer index Q.
        all_losses, all_indices = map(
            partial(torch.stack, dim=-1),
            (all_losses, all_indices)
        )

        ret = (quantized_out, all_indices, all_losses)

        if return_all_codes:
            # Reconstruct the actual code vectors for every chosen index
            #
            # Shape:
            #   sequence mode: [Q, B, N, D_codebook]
            #   image mode:    [Q, B, H, W, D_codebook]
            all_codes = self.get_codes_from_indices(all_indices)
            ret = (*ret, all_codes)

        return ret

## Model

In [53]:
from functools import reduce
from typing import Literal

import torch.nn as nn
# from vector_quantize_pytorch import ResidualVQ

from encoder import Encoder
from decoder import Decoder


class SoundStream(nn.Module):
    def __init__(self, n_q, codebook_size, D, C, strides=(2, 4, 5, 8)):
        super(SoundStream, self).__init__()

        # The temporal resampling ratio between input waveform and embeddings.
        # Not used in here, but helpful for consumers.
        self.M = reduce(lambda a, b: a * b, strides)

        self.encoder = Encoder(C=C, D=D, strides=strides)
        self.quantizer = ResidualVQ(
            num_quantizers=n_q,
            codebook_size=codebook_size,
            dim=D,
            kmeans_init=True,
            kmeans_iters=100,
            threshold_ema_dead_code=2
        )
        self.decoder = Decoder(C=C, D=D, strides=strides)

    def forward(
            self,
            x,
            mode: Literal['end-to-end', 'encode', 'decode'] = 'end-to-end',
        ):
        # x: batch_size x 1 x (T / 1)
        # e: batch_size x (T / M) x D --- where M is product of all numbers in `strides` tuple
        # o: batch_size x 1 x (T / 1)

        if mode == 'end-to-end':
            e = self.encoder(x)
            quantized, _, _ = self.quantizer(e.permute((0,2,1)))
            o = self.decoder(quantized.permute((0,2,1)))
            return o
        
        if mode == 'encode':
            e = self.encoder(x)
            quantized, _, _ = self.quantizer(e.permute((0,2,1)))
            return quantized
        
        if mode == 'decode':
            o = self.decoder(x.permute((0,2,1)))
            return o


In [54]:
import torch
import torchaudio
from huggingface_hub import hf_hub_download

# from soundstream.soundstream import SoundStream


def _infer_device():
    if torch.cuda.is_available():
        return 'cuda'
    elif torch.backends.mps.is_available():
        return 'mps'
    else:
        return 'cpu'


def from_pretrained(
    repo_id='haydenshively/SoundStream',
    filename='soundstream_variant_naturalspeech2.pt',
    device=None,
):
    if device is None:
        device = _infer_device()

    checkpoint_path = hf_hub_download(repo_id=repo_id, filename=filename)

    model_naturalspeech2 = SoundStream(
        n_q=16,
        codebook_size=1024,
        D=256,
        C=58,
        strides=(2, 4, 5, 5),
    )
    model_naturalspeech2.load_state_dict(
        torch.load(checkpoint_path, map_location=device)
    )

    return model_naturalspeech2


def load(waveform_path):
    # Load audio from file
    waveform, sample_rate = torchaudio.load(waveform_path)

    # Resample to the frequency the model was trained on
    resampler = torchaudio.transforms.Resample(
        sample_rate,
        16000,
        dtype=waveform.dtype
    )
    waveform = resampler(waveform)
    # Combine channels to get mono audio
    waveform = waveform.mean(dim=0, keepdim=True)

    return torch.unsqueeze(waveform, dim=0)


In [55]:
model = from_pretrained()

In [56]:
model.quantizer.__dir__()

['T_destination',
 '__annotations__',
 '__call__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_apply',
 '_backward_hooks',
 '_backward_pre_hooks',
 '_buffers',
 '_call_impl',
 '_compiled_call_impl',
 '_forward_hooks',
 '_forward_hooks_always_called',
 '_forward_hooks_with_kwargs',
 '_forward_pre_hooks',
 '_forward_pre_hooks_with_kwargs',
 '_get_backward_hooks',
 '_get_backward_pre_hooks',
 '_get_name',
 '_is_full_backward_hook',
 '_load_from_state_dict',
 '_load_state_dict_post_hooks',
 '_load_state_dict_pre_hooks',
 '_maybe_warn_non_full_backward_hook',
 '_modules',
 '_named_members',
 '_non_persistent_buffers_se

In [57]:
model.quantizer.codebooks.shape

torch.Size([16, 1024, 256])

In [58]:
load('00000.wav').shape

torch.Size([1, 1, 87320])

In [59]:
wav = model.forward(x=load('00000.wav'), mode='end-to-end').cpu().detach().numpy()

In [63]:
from IPython.display import display, Audio
display(Audio(wav[0], rate=16000))